# Mandate — the generated corpus, run with the internet off

Nothing about the kernel changes here: the same nine checks, the same seven
oracles, the same seed rule. What this notebook adds is **n** — a generated
corpus of 420 benign tasks and 735 cases per batch, against the hand-written
corpus's 25 and 105 — and a containment claim in its strong form, because a
session with `enable_internet: false` cannot reach a non-local host even if
the guard failed.

This file is committed to the repository. A hand-run notebook whose exact code
nobody kept is not a reproducible measurement.


## 1. Attach the repository, with no network

The repository is an attached dataset. It goes on `sys.path` rather than
through `pip install -e`: the attached directory is read-only, an editable
install would try to write `egg-info` into it, and the project is a flat
layout that `conftest.py` already puts on the path the same way.

Its two third-party dependencies are in the Kaggle image. Their versions are
**recorded, not assumed** — a signature check that silently ran against a
different `cryptography` is a number nobody could reproduce.


In [ ]:
import json, os, platform, sys, time, hashlib, zipfile, shutil
from pathlib import Path

# What is actually mounted, printed BEFORE anything is asserted.
#
# A missing attachment is the single most likely way this notebook fails, and
# the first version of this cell asserted on a path and said nothing else — so
# the log read 'mandate-repo is not there' and could not say what *was*. It
# turned out Kaggle nests attachments under datasets/<owner>/<slug> rather
# than mounting them flat at /kaggle/input/<slug>, which one line of output
# answered and a second guess would not have.
# The three roots, overridable by environment so this notebook can be
# **rehearsed locally**. Three of the four ways it has failed so far were
# properties of the paths — a mount layout, a read-only file system, an output
# directory that is also the upload — and none of them needed Kaggle to
# reproduce. A hosted run costs minutes and a push; a rehearsal costs seconds,
# and tests/test_kaggle_notebook.py runs these same cells against a fake mount.
INPUT = Path(os.environ.get('MANDATE_INPUT', '/kaggle/input'))
WORKING = Path(os.environ.get('MANDATE_WORKING', '/kaggle/working'))
SCRATCH = Path(os.environ.get('MANDATE_SCRATCH', '/tmp/mandate'))
WORKING.mkdir(parents=True, exist_ok=True)


def tree(root, depth=3, prefix=''):
    if depth == 0 or not root.is_dir():
        return
    for entry in sorted(root.iterdir())[:12]:
        print(f'{prefix}{entry.name}{"/" if entry.is_dir() else ""}')
        tree(entry, depth - 1, prefix + '  ')


print('/kaggle/input:')
tree(INPUT)

# Both layouts are searched rather than either assumed. Bounded depth, so this
# never walks a 38 MB catalogue looking for a directory.
ATTACHED = None
for pattern in ('mandate-repo', '*/mandate-repo', '*/*/mandate-repo'):
    found = [p for p in sorted(INPUT.glob(pattern)) if p.is_dir()]
    if found:
        ATTACHED = found[0]
        break
assert ATTACHED is not None, (
    'the repository dataset is not attached anywhere under /kaggle/input. '
    'The tree above is what the session actually has.'
)
print('repository dataset at', ATTACHED)

# **Copied to a writable place, not run from the mount.** /kaggle/input is a
# read-only file system, and this project writes beside itself in the ordinary
# course of a run: every kernel-arm case exports its audit chain, and the
# default path for that is `<repo>/runs/`. Running from the mount produced
# `OSError: [Errno 30] Read-only file system` a hundred cases in — late, and
# for a reason unrelated to anything being measured.
#
# Copying is a second or two for 3200 files and it makes the session identical
# to a local clone, which is the property worth having: nothing below has to
# remember to redirect a path. The corpus is verified by digest afterwards, so
# the copy is provably the same corpus.
#
# Into /tmp rather than /kaggle/working, because **everything in
# /kaggle/working is uploaded as the kernel's output**. A 3200-file copy of the
# repository in there would be pulled back down by `mk kaggle pull` and would
# bury the handful of files that are actually the measurement.
REPO = SCRATCH / 'repo'
shutil.rmtree(REPO, ignore_errors=True)
if (ATTACHED / 'mk.py').is_file():
    shutil.copytree(ATTACHED, REPO)
else:
    # Kaggle serves a `-r zip` upload extracted, but not always: a dataset of
    # a few thousand small files can arrive as one archive.
    archives = sorted(ATTACHED.glob('*.zip'))
    assert archives, (
        f'{ATTACHED} holds neither mk.py nor a zip; it holds '
        f'{sorted(p.name for p in ATTACHED.iterdir())[:10]}'
    )
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(REPO)
assert (REPO / 'mk.py').is_file(), f'{REPO} is not the repository'

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print('running from', REPO, '(writable)')

import cryptography, pydantic
ENV = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'machine': platform.machine(),
    'cryptography': cryptography.__version__,
    'pydantic': pydantic.VERSION,
    'attached': str(ATTACHED),
    'repo': str(REPO),
    'internet_enabled': False,
}
ENV

## 2. Verify the pinned datasets before reading a row

Kaggle mounts an attached dataset at `/kaggle/input/<slug>` — no owner, no
version in the path. That is exactly why the digest check matters more here,
not less: the path cannot tell you which version arrived and the digest can.

The corpus itself is committed, so this notebook does not regenerate it. The
check is here anyway, because a session whose datasets did not match the pin
is a session whose corpus nobody could re-derive.


In [ ]:
from harness.datasets import read_registry, verify

DATASETS = {}
for role, entry in sorted(read_registry().items()):
    differences = verify(role)
    DATASETS[role] = {'pin': entry.pin, 'digest': entry.digest,
                      'licence': entry.licence, 'ok': not differences}
    print(f"{role:<20} {entry.pin:<70} {'ok' if not differences else differences}")
DATASETS


## 3. Verify the corpus has not moved

Both manifests. An edit anywhere — a task, a case, a signed mandate, a shard —
fails here rather than producing a table nobody can attribute.


In [ ]:
from harness.manifest import current_hash, generated_hash, verify_all

drift = verify_all()
assert not drift, drift
CORPUS = {'handwritten': current_hash(), 'generated': generated_hash()}
CORPUS


## 4. Arm containment for the whole session

The platform already refuses the network. The guard is armed anyway, because
a platform setting and a run record are different evidence and `results.md`
quotes the second. The allowance is **empty**: with the deterministic
stand-in there is no endpoint to reach, so the claim is zero non-local
sockets rather than zero-except-one.

`harness.runner.run_case` arms the guard per run as well; this outer guard is
what makes the claim cover the whole session, including anything the notebook
itself does between suites.


In [ ]:
from harness.containment import contained

GUARD = contained(allow=())
CONTAINMENT_LOG = GUARD.__enter__()
CONTAINMENT_LOG.as_dict()


## 5. Measure the per-case cost, then choose the shard count

Chosen from a measurement rather than guessed. Kaggle's CPU sessions have a
wall-clock limit; the shard count is whatever keeps the largest shard inside
`BUDGET_S` at the cost this machine actually shows.

With the stand-in the per-case cost is milliseconds and this will pick one
shard, which is the honest answer: the sharding exists for the arm where a
case costs seconds, and a notebook that split a 40-second job into eight
would be measuring process startup.


In [ ]:
from harness.suite import DATASETS as SUITE_DATASETS, select
from harness.runner import run_case

WARMUP = 12          # cases timed before deciding
BUDGET_S = 8 * 3600  # the wall-clock a shard is allowed
ARMS = os.environ.get(
    'MANDATE_ARMS',
    'undefended,model-only,kernel,agent-guard,kernel+agent-guard',
).split(',')
# 'gen_b' is added only with a logged opening; see the closing note.
PLAN = os.environ.get('MANDATE_PLAN', 'gen_benign,gen_a').split(',')
LIMIT = int(os.environ.get('MANDATE_LIMIT', '0')) or None

sample = select(PLAN[-1])[:WARMUP]
start = time.monotonic()
for case in sample:
    run_case(case.task_id, attack_id=case.attack_id, config='kernel',
             model='scripted', export_chain=SCRATCH / 'warmup.chain.jsonl')
PER_CASE_S = (time.monotonic() - start) / len(sample)

TOTAL_CASES = sum(len(select(d, limit=LIMIT)) for d in PLAN) * len(ARMS)
SHARDS = max(1, -(-int(PER_CASE_S * TOTAL_CASES) // BUDGET_S))
{'per_case_s': round(PER_CASE_S, 4), 'total_cases': TOTAL_CASES,
 'estimated_s': round(PER_CASE_S * TOTAL_CASES, 1), 'shards': SHARDS}


## 6. Run the shards

`run_suite` refuses re-entry in one process — SQLite has a single writer and
the overhead column must not become a measurement of lock contention — so
shards run one after another here, each writing its own JSONL and its own
metadata. That is the same shape a separate process produces, and `mk merge`
reads them identically.

A shard is a contiguous block of the frozen corpus order, so shard 3 of 8 is
the same cases on any machine.


In [ ]:
from harness.shard import Shard
from harness.suite import run_suite

# Suites write into scratch, and only the two files per suite that *are* the
# measurement are copied into the output. A kernel-arm suite also exports one
# audit chain per case — 5775 of them here — and those are evidence the session
# holds rather than results: every line already carries its chain's head hash
# and entry count, and uploading twenty megabytes of chains would make the
# pull slow and the output unreadable.
RUN_DIR = SCRATCH / 'out'
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT = WORKING

written = []
for dataset in PLAN:
    for config in ARMS:
        for index in range(SHARDS):
            shard = Shard(index=index, count=SHARDS)
            cases = select(dataset, shard=shard, limit=LIMIT)
            stem = f"{dataset}.{config.replace('+','_')}.{shard.label}"
            result = run_suite(cases, dataset=dataset, config=config,
                               seed='0', model='scripted',
                               out=RUN_DIR / f'{stem}.jsonl', shard=shard)
            for produced in (result.path, result.meta_path):
                shutil.copy2(produced, OUT / produced.name)
                written.append(OUT / produced.name)
            print(f'{stem:<52} {len(result.scored):>4} scored  '
                  f'{result.attacker_wins:>4} wins  {len(result.errors)} error(s)')
len(written)


## 7. Close the guard and write the digests

`digests.json` is what `mk kaggle pull` checks before the merge step is
allowed to see any of this. Without it a truncated upload is a file that
parses — and a table short one shard looks exactly like a table of a smaller
suite.


In [ ]:
GUARD.__exit__(None, None, None)
CONTAINMENT = CONTAINMENT_LOG.as_dict()
assert CONTAINMENT['non_local_blocked'] == 0, CONTAINMENT
assert CONTAINMENT['non_local_allowed'] == 0, CONTAINMENT

files = {}
for path in sorted(set(written)):
    raw = path.read_bytes()
    files[path.name] = {'sha256': 'sha256:' + hashlib.sha256(raw).hexdigest(),
                        'bytes': len(raw)}

index = {
    'run': {'env': ENV, 'corpus': CORPUS, 'datasets': DATASETS,
            'arms': ARMS, 'plan': PLAN, 'seed': '0', 'model': 'scripted',
            'per_case_s': round(PER_CASE_S, 4),
            'session_containment': CONTAINMENT},
    'shards': sorted({p.name.rsplit('.', 2)[-2] for p in written if p.suffix == '.jsonl'}),
    'files': files,
}
(OUT / 'digests.json').write_text(json.dumps(index, indent=2, sort_keys=True) + '\n')
print(json.dumps(index['run'], indent=2, sort_keys=True))
print(f"{len(files)} file(s), {sum(f['bytes'] for f in files.values()) / 1e6:.1f} MB")


## What this run does not say

* **Zero non-local sockets is a statement about Python's `socket` module.**
  A subprocess or a C extension holding its own descriptor would go around
  the guard. Nothing on the run path does either, and a hosted runner does
  not make this a sandbox.
* **The stand-in drove it.** `scripted-gullible-v1` is a rule-based planner,
  not a model, so no ASR figure from this run is a model measurement.
* **`gen-b` is not in `PLAN` above.** It is held out; running it needs
  `harness.corpus.open_batch('gen-b', reason=...)`, and the opening is logged
  to `harness/attacks/openings.jsonl` — which cannot be written from a
  read-only attached dataset, so a held-out run is taken on the machine that
  keeps the log.
